# 01. RNA 한 표본과 약물 민감도 정답 읽기

이 노트북은 학습을 하지 않습니다. 이미 만들어 둔 입력 캐시를 읽어, 한 세포 모델의 처리 전 RNA와 그 모델에 대해 측정된 약물 민감도(Activity Area, AA)가 어떻게 생겼는지 봅니다.

먼저 서버 작업 폴더에서 `sbatch experiments/pathway_sensitivity_a1/a0_build.sbatch`가 한 번 실행되어 있어야 합니다. 노트북 커널은 `assets/envs/psa1` 환경을 사용합니다.

In [ ]:
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "configs", "pathway_sensitivity_a1")):
    parent = os.path.dirname(root)
    assert parent != root, "저장소 안에서 실행해 주세요."
    root = parent
sys.path.insert(0, os.path.join(root, "src", "pathway_sensitivity_a1"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import core
cfg = core.load_config()
print("저장소:", root)
print("입력 캐시:", core.cache_dir(cfg))

## 1. 어떤 세포 모델이 분석에 들어가는가

`model_registry.tsv`는 모델마다 연구 이름, 진단, 포함 여부, 계열(`family_id`)을 적어 둔 표입니다. 같은 환자에서 나온 모델은 같은 계열로 묶여 항상 같은 fold에 들어갑니다.

In [ ]:
reg = core.load_registry()
lim = reg[reg.study == "LIMORE"]
print("LIMORE 모델:", len(lim), "/ 포함:", int(lim.included.sum()), "/ 계열:", lim[lim.included].family_id.nunique())
print(lim[~lim.included][["source_name", "exclusion_reason"]].to_string(index=False))
multi = lim[lim.included].groupby("family_id").source_name.apply(list)
print(multi[multi.map(len) > 1])

## 2. 한 표본의 RNA

행은 세포 모델, 열은 유전자입니다. 값은 이미 log2로 변환된 FPKM입니다. 다시 로그를 취하지 않습니다.

In [ ]:
limore = core.load_study(cfg, "LIMORE")
axis = pd.read_csv(os.path.join(core.MANIFESTS, "gene_axis.tsv"), sep="\t")
print("행렬 크기 (모델 x 유전자):", limore.expr.shape, " 최솟값", limore.expr.min(), " 최댓값", limore.expr.max())
m = "LIMORE__Huh7"
i = limore.models.index(m)              # 이름으로 행을 찾습니다. 순서를 가정하지 않습니다.
one = pd.Series(limore.expr[i], index=axis.symbol)
print(one.loc[["ALB", "AFP", "GPC3", "TP53", "MKI67"]])

## 3. 그 표본의 정답: 약물별 AA

AA가 클수록 그 약물에 민감합니다. 방향은 자료에서 확인합니다. AA가 큰 쌍일수록 IC50과 Emax(최대 농도에서 남은 생존 비율)가 작아야 합니다.

In [ ]:
resp = core.load_limore_response(cfg)
mine = resp[resp.model_id == m].sort_values("y", ascending=False)
print(mine.head(5).to_string(index=False))
from scipy.stats import spearmanr
print("AA-IC50 Spearman:", round(spearmanr(resp.y, resp.IC50)[0], 3), " AA-Emax Spearman:", round(spearmanr(resp.y, resp.Emax)[0], 3))

## 4. 한 약물의 AA 분포

약물마다 값의 범위가 다릅니다. 그래서 평가는 약물별로 따로 하고, 학습 자료의 표준편차로 나눈 오차를 씁니다.

In [ ]:
drug = "Sorafenib"
elig = set(lim[lim.included].model_id)
y = resp[(resp.drug_id == drug) & resp.model_id.isin(elig)].y
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(y, bins=20, color="#2a78d6")
ax.set_xlabel(f"{drug} Activity Area"); ax.set_ylabel("models")
print("모델", len(y), " 평균", round(y.mean(), 3), " 표준편차", round(y.std(), 3))